In [38]:
# LING 539 Term Project - Text Classification
## Skye Lewis
## May 2026

In [9]:
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from typing import Iterator, Iterable, List, Tuple, Text, Union

In [41]:
class TextToFeatures:
    """
        Initializes an object for converting texts to features.    
    """
    
    PUNCTUATION = r"""[,.?!:;'"`‛'""]"""

    def __init__(self, **vectorizer_kwargs):
        self.vectorizer = CountVectorizer(**vectorizer_kwargs)
        self._vocab = None

    def _preprocess_text(self, text):
        """
            Lowercase, strip HTML tags, and remove punctuation.
        """
        text = text.lower()
        text = re.sub(r"<[^>]+>", " ", text)
        text = re.sub(TextToFeatures.PUNCTUATION, ' ', text)
#         text = re.sub(r"[^a-z0-9\s]", " ", text)
        return text

    def _fit(self, texts):
        """
            Fits ("trains") a TextToFeature instance on a collection of documents.
        
            :param training_texts: The training texts.
        """
        cleaned = [self._preprocess_text(t) for t in texts]
        self.vectorizer.fit(cleaned)
        self._vocab = self.vectorizer.vocabulary_
        return self

    def index(self, word) -> Union[None, int]:
        """
            Return the feature index for a given word, or None if not present.
            
            :param texts: A sequence of texts.
            :return: The unique integer index associated with the feature or None if not present.
        """
        return self._vocab.get(self._preprocess_text(word))

    def transform(self, texts):
        """
            Creates a feature matrix from a sequence of texts.
        
            :param texts: A sequence of texts.
            :return: A matrix, with one row of feature values for each text.
        """
        cleaned = [self._preprocess_text(t) for t in texts]
        return self.vectorizer.transform(cleaned)

In [42]:
class TextToLabels:
    """
        Initializes an object for converting texts to labels.
    """

    def __init__(self):
        self.encoder = LabelEncoder()
        self._classes = None

    def fit(self, labels):
        """
            Assigns each distinct label a unique integer.
        
            :param labels: The training labels.
        """
        self.encoder.fit(labels)
        self._classes = set(self.encoder.classes_)
        return self

    def index(self, label):
        """
            Returns the index in the vocabulary of the given label.

            :param label: A label
            :return: The unique integer index associated with the label.
        """
        return int(self.encoder.transform([label])[0])

    def transform(self, labels):
        """
            Creates a label vector from a sequence of labels.

            Each entry in the vector corresponds to one of the input labels. The
            value at index j is the unique integer associated with the jth label.

            :param labels: A sequence of labels.
            :return: A vector, with one entry for each label.
        """
        return self.encoder.transform(labels)

    def __contains__(self, label):
        """
            Special "dunder" method to check if a label is known to the TextToLabels instance.
        
            :return: True if the label was seen in the training data; False otherwise
        """
        return label in self._classes

In [43]:
class Classifier:
    """
        Initalizes a logistic regression classifier.
    """

    def __init__(self, **logistic_regression_kwargs):
        self.featurizer = TextToFeatures()
        self.labeler = TextToLabels()
        self.model = LogisticRegression(**logistic_regression_kwargs)

    def train(self, texts, labels):
        """
            Trains the classifier using the given training examples.      
        """
        self.featurizer._fit(texts)
        self.labeler.fit(labels)
        X = self.featurizer.transform(texts)
        y = self.labeler.transform(labels)
        self.model.fit(X, y)
        return self

    def predict(self, texts):
        """
            Makes predictions for each of the given examples.
        """
        X = self.featurizer.transform(texts)
        return self.model.predict(X)

In [44]:
df = pd.read_csv("train.csv")

texts = df["TEXT"].astype(str).tolist()
labels = df["LABEL"].tolist()

X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.2, random_state=11, stratify=labels
)

In [45]:
clf = Classifier(max_iter=2000, solver="lbfgs", multi_class="multinomial")
clf.train(X_train, y_train)

In [46]:
y_pred = clf.predict(X_val)
y_train_ind = clf.labeler.transform(y_val)

f1_val = f1_score(y_train_ind, y_pred, average="macro")
f1_per_class = f1_score(y_train_ind, y_pred, average=None)

print(f"F1:  {f1_val:.1%}")

F1:  90.2%


In [47]:
test_df = pd.read_csv("test.csv")
test_texts = test_df["TEXT"].astype(str).tolist()

test_preds = clf.predict(test_texts)

output_df = pd.DataFrame({"ID": test_df["ID"], "LABEL": test_preds})
output_df.to_csv("predictions.csv", index=False)
print(f"Wrote {len(output_df)} predictions")
print(output_df["LABEL"].value_counts())

Wrote 17580 predictions
0    8093
1    4884
2    4603
Name: LABEL, dtype: int64
